In [1]:
import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

In [2]:
from utils.tokenizer import prepare_tokenizer, pad_tensor
from utils.load_data import load_data, prep_dolly_collate_fn, prep_squad_collate_fn
from utils.checkpoint import try_loading, save_checkpoint

In [3]:
from models.model import VanillaEncoderDecoder

In [4]:
from tqdm.notebook import tqdm

In [5]:
vocab, eos_idx, bos_idx, pad_idx, a_pad_idx, vocab_size = prepare_tokenizer()
squad, dolly = load_data()

In [6]:
qc_len = 512 + 256
a_len = 100

In [7]:
squad_collate = prep_squad_collate_fn(qc_len, a_len, eos_idx, bos_idx, pad_idx, a_pad_idx, pad_tensor)

In [8]:
batch_size = 24
embedding_dim = 512
num_heads = 4
phm_factor = 4
lm_head_factor = 2
num_encoder_layers = 4
num_decoder_layers = 3

In [9]:
squadDataloader = DataLoader(squad, batch_size=batch_size, shuffle=True, collate_fn=squad_collate)
squadIter = iter(squadDataloader)

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
from torch.optim import Adam

In [12]:
# create_model_fallback_fn is used by checkpoint loader in the event
# that the checkpoint file was not found (e.g. first run)
def create_model(embedding_dim, num_heads, num_encoder_layers, num_decoder_layers, vocab_size, phm_factor, lm_head_factor, eos_idx, bos_idx, qc_pad_idx, a_pad_idx, device):
    model = VanillaEncoderDecoder(embedding_dim, num_heads, num_encoder_layers, num_decoder_layers, vocab_size, phm_factor, lm_head_factor, eos_idx, bos_idx, qc_pad_idx, a_pad_idx)
    model.to(device)
    optm = Adam(model.parameters(), lr=1e-4)
    return model, optm

create_model_fallback_fn = lambda: create_model(embedding_dim, num_heads, num_encoder_layers, num_decoder_layers, vocab_size, phm_factor, lm_head_factor, eos_idx, bos_idx, pad_idx, a_pad_idx, device)

In [13]:
checkpoint_dir = "./checkpoints"
checkpoint_name = "vanilla_encoder_decoder.pt"
model_class = VanillaEncoderDecoder
optm_class = Adam

In [14]:
model, optm, losses, log_dir = try_loading(checkpoint_dir, checkpoint_name, model_class, optm_class, device, create_model_fallback_fn)

Resuming, have seen 9000 epochs
Have 13247880 trainable parameters
Logging to runs/run_at_2023-11-30_20-59-23


In [15]:
writer = SummaryWriter(log_dir=log_dir)

In [16]:
total_epochs = 10000
seen_epochs = len(losses)
remaining_epochs = total_epochs - seen_epochs
save_every = 5

In [17]:
from torch.nn.functional import cross_entropy

In [18]:
for epoch in (pbar := tqdm(range(remaining_epochs))):
    try:
        batch = next(squadIter)
    except StopIteration:
        squadIter = iter(squadDataloader)
        batch = next(squadIter)
    questioncontext, answer = batch
    a_input, a_target = answer[:, :-1], answer[:, 1:]
    questioncontext, a_input, a_target = questioncontext.to(device), a_input.to(device), a_target.to(device)
    optm.zero_grad()
    logits = model(questioncontext, a_input)
    loss = cross_entropy(logits.reshape(-1, vocab_size), a_target.reshape(-1), ignore_index=a_pad_idx)
    loss.backward()
    optm.step()
    losses.append(loss.item())
    writer.add_scalar("loss", loss.item(), epoch + seen_epochs)
    pbar.set_description(f"loss: {loss.item()}")
    if epoch % save_every == 0:
        save_checkpoint(checkpoint_dir, checkpoint_name, model, losses, optm, tensorboard_log_dir=log_dir)

  0%|          | 0/1000 [00:00<?, ?it/s]

In [19]:
save_checkpoint(checkpoint_dir, checkpoint_name, model, losses, optm, tensorboard_log_dir=log_dir)

In [20]:
# take a random sample from the squad dataset and see how the model performs
sample_i = 3
sample = squad[sample_i]
context = sample["context"]
question = sample["question"]
answer = sample["answers"]

In [21]:
# combine question and context so question comes first
qc = torch.cat([question, context])

In [22]:
qc_text = vocab.decode(qc.tolist())

In [23]:
a_text = vocab.decode(answer.tolist())

In [24]:
# now pad qc
qc = pad_tensor(qc, qc_len, eos_idx, bos_idx, pad_idx)

In [25]:
qc = qc.to(device)

In [26]:
# greedy decoding
def greedy_decode(model, qc, bos_idx, eos_idx, max_len, starting=[bos_idx]):
    model.eval()
    with torch.no_grad():
        qc = qc.unsqueeze(0)
        a = torch.tensor(starting).unsqueeze(0).to(qc.device)
        for i in range(max_len):
            logits = model(qc, a)
            logits = logits[0, -1, :]
            logits = logits.softmax(dim=-1)
            next_token = logits.argmax()
            a = torch.cat([a, next_token.unsqueeze(0).unsqueeze(0)], dim=1)
            if next_token == eos_idx:
                break
    return a.squeeze(0)

In [27]:
decoded = greedy_decode(model, qc, bos_idx, eos_idx, a_len)

In [28]:
qc_text

'What is the Grotto at Notre Dame?Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [29]:
a_text

'a Marian place of prayer and reflection'

In [30]:
decoded

tensor([8001,  332,    3, 1060,  443, 8000], device='cuda:0')

In [31]:
vocab.decode(decoded.tolist())

'Italy'